# 🚀 SURESH.GPT Platform on Google Colab (Free T4 / A100 GPU)

Run your entire **SURESH.GPT Platform** with full GPU hardware acceleration (30–50 tokens/sec) on Google Colab.

### Step 1: Ensure T4 GPU is Enabled
1. Go to top menu: **Runtime** > **Change runtime type**
2. Select **T4 GPU** > Click **Save**
3. Run the cell below to verify:

In [ ]:
!nvidia-smi

### Step 2: Unzip Project Files
Upload `LLM_XRAY_PROJECT_COLAB.zip` to the Colab files panel (📁 icon on the left), then run this cell:

In [ ]:
import os

zip_name = 'LLM_XRAY_PROJECT_COLAB.zip'
if not os.path.exists(f'/content/{zip_name}'):
    if os.path.exists('/content/LLM-XRAY_PROJECT.zip'):
        zip_name = 'LLM-XRAY_PROJECT.zip'
    else:
        raise FileNotFoundError("Please upload 'LLM_XRAY_PROJECT_COLAB.zip' using the left file panel.")

!mkdir -p /content/LLM_XRAY_PROJECT
!unzip -q -o /content/{zip_name} -d /content/LLM_XRAY_PROJECT
%cd /content/LLM_XRAY_PROJECT
!ls -la

### Step 3: Install Required Dependencies & Tunnel Tools

In [ ]:
!pip install -q streamlit faiss-cpu sentence-transformers pypdf python-docx firebase-admin python-dotenv streamlit-option-menu sqlalchemy reportlab accelerate
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1
print("✅ All dependencies installed successfully!")

### Step 4: Configure Streamlit & Start GPU Server
This creates the required headless configuration so Streamlit starts instantly without prompting for an email or hanging:

In [ ]:
import time
import urllib.request

# 1. Configure Streamlit headless mode
!mkdir -p ~/.streamlit
with open(os.path.expanduser('~/.streamlit/config.toml'), 'w') as f:
    f.write('''[server]
headless = true
port = 8501
address = "0.0.0.0"
enableCORS = false
enableXsrfProtection = false

[browser]
gatherUsageStats = false
''')

# 2. Kill any old streamlit processes if already running
!pkill -f streamlit || true

# 3. Start Streamlit in background and stream logs
!nohup streamlit run app.py --server.port 8501 --server.address 0.0.0.0 > /content/streamlit.log 2>&1 &

# 4. Health-check loop (Wait until server is ready)
print("⏳ Waiting for Streamlit server to start on GPU...")
server_ready = False
for i in range(25):
    try:
        urllib.request.urlopen("http://127.0.0.1:8501", timeout=1)
        server_ready = True
        print("✅ Streamlit is UP and actively serving on port 8501!")
        break
    except Exception:
        time.sleep(1)

if not server_ready:
    print("❌ Streamlit failed to start. Here are the logs:")
    !cat /content/streamlit.log

### Step 5: Open Application (Choose Option A or Option B)

#### Option A (Recommended): Official Cloudflare Tunnel (No password needed)
Run the cell below. It will output a public link `https://*.trycloudflare.com`.
Click the link to open the app!

In [ ]:
!cloudflared tunnel --url http://127.0.0.1:8501

#### Option B: Google Colab Native Proxy
If you prefer to open the app directly inside Google Colab without external tunnels, run this cell:

In [ ]:
from google.colab.output import serve_kernel_port_as_window
serve_kernel_port_as_window(8501)